# What concepts did we actually record?

In [1]:
from edgar import get_filings, set_identity, Company

# Essential for EDGAR access
set_identity("Luke Rast luke.rast@gmail.com")


In [2]:
company = Company("AAPL")

In [3]:
import polars as pl
import pandas as pd

In [12]:
single_sample = pd.concat([company.balance_sheet().to_dataframe(), company.income_statement().to_dataframe(), company.cash_flow().to_dataframe()])
single_sample = single_sample.reset_index()

In [15]:
labels_single = single_sample['label'].unique()
concepts_single = single_sample['concept'].unique()

In [6]:
all_data = pl.read_parquet('data/sheets.parquet')

In [23]:
print('all labels - single labels: ', len(set(all_data['label'].unique()) - set(labels_single)), 'total: ', len(all_data['label'].unique()))
print('all concepts - single concepts: ', len(set(all_data['concept'].unique()) - set(concepts_single)), 'total: ', len(all_data['concept'].unique()))

all labels - single labels:  270 total:  336

all concepts - single concepts:  247 total:  319

There are many labels and concepts that dont appear in individual companies sheets

#### Can we group by co-occurance?

In other words, group labels and concepts so that any co-occurance of the same label/concept groups them together.

In [59]:
all_labels = list(all_data['label'].unique())
all_concepts = list(all_data['concept'].unique())

def co_occuring_concepts(label):
    return all_data.filter(pl.col('label') == label)['concept'].unique()

def co_occuring_labels(concept):
    return all_data.filter(pl.col('concept') == concept)['label'].unique()

class closed_set():
    def __init__(self):
        self.labels = []
        self.concepts = []
    def add_label(self, label):
        if label in self.labels:
            return
        self.labels.append(label)
        for concept in co_occuring_concepts(label):
            
            self.add_concept(concept)

    def add_concept(self, concept):
        if concept in self.concepts:
            return
        self.concepts.append(concept)
        for label in co_occuring_labels(concept):
            self.add_label(label)



In [67]:
first_set = closed_set()
first_set.add_label(all_labels[0])

sets = []

for label in all_labels:
    seen = False
    for cSet in sets:
        if label in cSet.labels:
            seen = True
    if seen:
        continue
    new_set = closed_set()
    new_set.add_label(label)
    sets.append(new_set)



In [69]:
len(sets)

304

In [75]:
len(all_labels)

336

In [76]:
len(all_concepts)

319

This does not majorly change the outlook: there are still many possible datapoints that we could ask for!